<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebooks/boosting_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')

!kaggle datasets download -d mlg-ulb/creditcardfraud -p /content/data/ulb --unzip
!kaggle datasets download -d kartik2112/fraud-detection -p /content/data/sparkov --unzip


Kaggle username: Subbareddy_Subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100% 66.0M/66.0M [00:00<00:00, 140MB/s]

Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0
100% 202M/202M [00:01<00:00, 148MB/s]



In [2]:

# 2. Clone the repo (already has split.py and utils.py inside it)
github_token = getpass('Paste your GitHub token: ')
repo_url = f'https://{github_token}@github.com/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-.git'
!git clone {repo_url}


Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 139 (delta 85), reused 18 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 408.12 KiB | 4.58 MiB/s, done.
Resolving deltas: 100% (85/85), done.


In [3]:

# 3. Move in, confirm, move data in
%cd Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-
!mv /content/data/ulb /content/data/sparkov data/



/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


In [4]:
import sys
sys.path.append('/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-')
from src.split import split_sparkov, split_ulb

sparkov_train, sparkov_test = split_sparkov()
ulb_train, ulb_test = split_ulb()

Train: 1,296,675 rows, from 2019-01-01 00:00:18 to 2020-06-21 12:13:37
Test:  555,719 rows, from 2020-06-21 12:14:25 to 2020-12-31 23:59:34
Train: 226,980 rows, from 0.0 to 145233.0
Test:  56,746 rows, from 145234.0 to 172792.0


In [6]:
import inspect

print(inspect.signature(fit_category_risk))
print(inspect.signature(apply_category_risk))
print(inspect.signature(add_prev_txn_features))

(train_df, category_column, label_column)
(df, category_column, fitted)
(df, id_column, time_column, lat_column, lon_column)


In [7]:
# category risk: fit on train only, apply to both splits
category_risk_map = fit_category_risk(sparkov_train, 'category', 'is_fraud')
sparkov_train['category_risk'] = apply_category_risk(sparkov_train, 'category', category_risk_map)
sparkov_test['category_risk']  = apply_category_risk(sparkov_test, 'category', category_risk_map)

# implied velocity: built on the full train+test timeline so boundary
# transactions still have a real previous transaction to compare against
full = pd.concat([sparkov_train, sparkov_test]).sort_values('trans_date_trans_time')
full = add_prev_txn_features(full, 'cc_num', 'trans_date_trans_time', 'merch_lat', 'merch_long')

print(full.columns.tolist())

['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'category_risk', 'dist_from_prev_txn_km', 'time_gap_hours', 'implied_velocity_kmh']


In [8]:
# split back into train/test using the original index
sparkov_train = full.loc[sparkov_train.index]
sparkov_test  = full.loc[sparkov_test.index]

# first-ever transaction for a card has nothing to compare to -> 0
sparkov_train['implied_velocity_kmh'] = sparkov_train['implied_velocity_kmh'].fillna(0)
sparkov_test['implied_velocity_kmh']  = sparkov_test['implied_velocity_kmh'].fillna(0)

# velocity has extreme outliers, so log-transform it
sparkov_train['velocity_log'] = np.log1p(sparkov_train['implied_velocity_kmh'])
sparkov_test['velocity_log']  = np.log1p(sparkov_test['implied_velocity_kmh'])

# is_night: 10pm-3am, from the EDA
sparkov_train['hour'] = pd.to_datetime(sparkov_train['trans_date_trans_time']).dt.hour
sparkov_test['hour']  = pd.to_datetime(sparkov_test['trans_date_trans_time']).dt.hour
sparkov_train['is_night'] = sparkov_train['hour'].isin([22, 23, 0, 1, 2, 3]).astype(int)
sparkov_test['is_night']  = sparkov_test['hour'].isin([22, 23, 0, 1, 2, 3]).astype(int)

# amt_log: raw amount is heavily skewed
sparkov_train['amt_log'] = np.log1p(sparkov_train['amt'])
sparkov_test['amt_log']  = np.log1p(sparkov_test['amt'])

# age in years, from dob
sparkov_train['age'] = (pd.to_datetime(sparkov_train['trans_date_trans_time']) - pd.to_datetime(sparkov_train['dob'])).dt.days / 365.25
sparkov_test['age']  = (pd.to_datetime(sparkov_test['trans_date_trans_time'])  - pd.to_datetime(sparkov_test['dob'])).dt.days / 365.25

# gender_flag: 1 = male, 0 = female
sparkov_train['gender_flag'] = (sparkov_train['gender'] == 'M').astype(int)
sparkov_test['gender_flag']  = (sparkov_test['gender'] == 'M').astype(int)

# final six-feature matrices
features = ['amt_log', 'category_risk', 'is_night', 'velocity_log', 'age', 'gender_flag']
target = 'is_fraud'

X_train, y_train = sparkov_train[features], sparkov_train[target]
X_test, y_test = sparkov_test[features], sparkov_test[target]

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(1852394, 6) (1111438, 6)
0.005210014716091717 0.004845974314356716


In [9]:
# reset to a clean split (previous split-back step corrupted these)
sparkov_train, sparkov_test = split_sparkov()

# category risk: fit on train only, apply to both splits
category_risk_map = fit_category_risk(sparkov_train, 'category', 'is_fraud')
sparkov_train['category_risk'] = apply_category_risk(sparkov_train, 'category', category_risk_map)
sparkov_test['category_risk']  = apply_category_risk(sparkov_test, 'category', category_risk_map)

# mark which split each row came from, since train/test indexes overlap
sparkov_train['_split'] = 'train'
sparkov_test['_split']  = 'test'

# implied velocity: built on the full train+test timeline so boundary
# transactions still have a real previous transaction to compare against
full = pd.concat([sparkov_train, sparkov_test]).sort_values('trans_date_trans_time')
full = add_prev_txn_features(full, 'cc_num', 'trans_date_trans_time', 'merch_lat', 'merch_long')

# split back using the marker column, not the index
sparkov_train = full[full['_split'] == 'train'].drop(columns='_split')
sparkov_test  = full[full['_split'] == 'test'].drop(columns='_split')

print(sparkov_train.shape, sparkov_test.shape)

Train: 1,296,675 rows, from 2019-01-01 00:00:18 to 2020-06-21 12:13:37
Test:  555,719 rows, from 2020-06-21 12:14:25 to 2020-12-31 23:59:34
(1296675, 27) (555719, 27)


In [10]:
# velocity has extreme outliers, so log-transform it
# (first-ever transaction for a card has nothing to compare to -> 0)
sparkov_train['implied_velocity_kmh'] = sparkov_train['implied_velocity_kmh'].fillna(0)
sparkov_test['implied_velocity_kmh']  = sparkov_test['implied_velocity_kmh'].fillna(0)

sparkov_train['velocity_log'] = np.log1p(sparkov_train['implied_velocity_kmh'])
sparkov_test['velocity_log']  = np.log1p(sparkov_test['implied_velocity_kmh'])

# is_night: 10pm-3am, from the EDA
sparkov_train['hour'] = pd.to_datetime(sparkov_train['trans_date_trans_time']).dt.hour
sparkov_test['hour']  = pd.to_datetime(sparkov_test['trans_date_trans_time']).dt.hour
sparkov_train['is_night'] = sparkov_train['hour'].isin([22, 23, 0, 1, 2, 3]).astype(int)
sparkov_test['is_night']  = sparkov_test['hour'].isin([22, 23, 0, 1, 2, 3]).astype(int)

# amt_log: raw amount is heavily skewed
sparkov_train['amt_log'] = np.log1p(sparkov_train['amt'])
sparkov_test['amt_log']  = np.log1p(sparkov_test['amt'])

# age in years, from dob
sparkov_train['age'] = (pd.to_datetime(sparkov_train['trans_date_trans_time']) - pd.to_datetime(sparkov_train['dob'])).dt.days / 365.25
sparkov_test['age']  = (pd.to_datetime(sparkov_test['trans_date_trans_time'])  - pd.to_datetime(sparkov_test['dob'])).dt.days / 365.25

# gender_flag: 1 = male, 0 = female
sparkov_train['gender_flag'] = (sparkov_train['gender'] == 'M').astype(int)
sparkov_test['gender_flag']  = (sparkov_test['gender'] == 'M').astype(int)

# final six-feature matrices
features = ['amt_log', 'category_risk', 'is_night', 'velocity_log', 'age', 'gender_flag']
target = 'is_fraud'

X_train, y_train = sparkov_train[features], sparkov_train[target]
X_test, y_test = sparkov_test[features], sparkov_test[target]

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(1296675, 6) (555719, 6)
0.005788651743883394 0.0038598644278853163


In [11]:
from sklearn.model_selection import train_test_split

# carve a validation set out of train, for threshold tuning later
# stratify to keep the (rare) fraud rate consistent across both pieces
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

print(X_train_sub.shape, X_val.shape, X_test.shape)
print(y_train_sub.mean(), y_val.mean(), y_test.mean())

(1037340, 6) (259335, 6) (555719, 6)
0.00578884454470087 0.005787880540613492 0.0038598644278853163


In [12]:
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

# XGBoost, Control: no imbalance handling
xgb_control = XGBClassifier(
    n_estimators=100,
    eval_metric='logloss',
    random_state=42
)
xgb_control.fit(X_train_sub, y_train_sub)

val_probs_xgb_control = xgb_control.predict_proba(X_val)[:, 1]

pr_auc = average_precision_score(y_val, val_probs_xgb_control)
print('XGBoost Control - PR-AUC (val):', pr_auc)

XGBoost Control - PR-AUC (val): 0.8100456121914881


In [13]:
# XGBoost, with scale_pos_weight: weights the minority (fraud) class
# scale_pos_weight = (# negative) / (# positive), the standard XGBoost recommendation
neg, pos = (y_train_sub == 0).sum(), (y_train_sub == 1).sum()
spw = neg / pos
print('scale_pos_weight:', spw)

xgb_weighted = XGBClassifier(
    n_estimators=100,
    eval_metric='logloss',
    scale_pos_weight=spw,
    random_state=42
)
xgb_weighted.fit(X_train_sub, y_train_sub)

val_probs_xgb_weighted = xgb_weighted.predict_proba(X_val)[:, 1]

pr_auc = average_precision_score(y_val, val_probs_xgb_weighted)
print('XGBoost Weighted - PR-AUC (val):', pr_auc)

scale_pos_weight: 171.74604496253122
XGBoost Weighted - PR-AUC (val): 0.8715812257759131


In [14]:
from lightgbm import LGBMClassifier

# LightGBM, Control: no imbalance handling
lgbm_control = LGBMClassifier(
    n_estimators=100,
    random_state=42
)
lgbm_control.fit(X_train_sub, y_train_sub)

val_probs_lgbm_control = lgbm_control.predict_proba(X_val)[:, 1]

pr_auc = average_precision_score(y_val, val_probs_lgbm_control)
print('LightGBM Control - PR-AUC (val):', pr_auc)


[LightGBM] [Info] Number of positive: 6005, number of negative: 1031335
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022309 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 784
[LightGBM] [Info] Number of data points in the train set: 1037340, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.005789 -> initscore=-5.146017
[LightGBM] [Info] Start training from score -5.146017
LightGBM Control - PR-AUC (val): 0.6884606280962148


In [15]:
# LightGBM, with scale_pos_weight (same idea as XGBoost's)
lgbm_weighted = LGBMClassifier(
    n_estimators=100,
    scale_pos_weight=spw,
    random_state=42
)
lgbm_weighted.fit(X_train_sub, y_train_sub)

val_probs_lgbm_weighted = lgbm_weighted.predict_proba(X_val)[:, 1]

pr_auc = average_precision_score(y_val, val_probs_lgbm_weighted)
print('LightGBM Weighted - PR-AUC (val):', pr_auc)

[LightGBM] [Info] Number of positive: 6005, number of negative: 1031335
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.204920 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 784
[LightGBM] [Info] Number of data points in the train set: 1037340, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.005789 -> initscore=-5.146017
[LightGBM] [Info] Start training from score -5.146017
LightGBM Weighted - PR-AUC (val): 0.14176850727638243


In [16]:
# sweep scale_pos_weight to see where LightGBM's PR-AUC breaks down
weight_options = [1, 5, 20, 50, 100, spw]

for w in weight_options:
    model = LGBMClassifier(n_estimators=100, scale_pos_weight=w, random_state=42, verbose=-1)
    model.fit(X_train_sub, y_train_sub)
    probs = model.predict_proba(X_val)[:, 1]
    pr_auc = average_precision_score(y_val, probs)
    print(f'scale_pos_weight={w:.2f} -> PR-AUC: {pr_auc:.4f}')

# also try LightGBM's own built-in flag instead of an explicit weight
model_unbalance = LGBMClassifier(n_estimators=100, is_unbalance=True, random_state=42, verbose=-1)
model_unbalance.fit(X_train_sub, y_train_sub)
probs_unbalance = model_unbalance.predict_proba(X_val)[:, 1]
pr_auc_unbalance = average_precision_score(y_val, probs_unbalance)
print(f'is_unbalance=True -> PR-AUC: {pr_auc_unbalance:.4f}')

scale_pos_weight=1.00 -> PR-AUC: 0.6885
scale_pos_weight=5.00 -> PR-AUC: 0.6003
scale_pos_weight=20.00 -> PR-AUC: 0.2594
scale_pos_weight=50.00 -> PR-AUC: 0.1686
scale_pos_weight=100.00 -> PR-AUC: 0.1517
scale_pos_weight=171.75 -> PR-AUC: 0.1418
is_unbalance=True -> PR-AUC: 0.1418


In [17]:
from sklearn.metrics import fbeta_score

def best_f2_threshold(y_true, probs):
    thresholds = np.linspace(0.01, 0.99, 99)
    best_thresh, best_f2 = 0.5, 0
    for t in thresholds:
        preds = (probs >= t).astype(int)
        f2 = fbeta_score(y_true, preds, beta=2, zero_division=0)
        if f2 > best_f2:
            best_f2, best_thresh = f2, t
    return best_thresh, best_f2

models_val_probs = {
    'XGBoost Control': val_probs_xgb_control,
    'XGBoost Weighted': val_probs_xgb_weighted,
    'LightGBM Control': val_probs_lgbm_control,
}

thresholds = {}
for name, probs in models_val_probs.items():
    t, f2 = best_f2_threshold(y_val, probs)
    thresholds[name] = t
    print(f'{name}: best threshold = {t:.2f}, F2 (val) = {f2:.4f}')

XGBoost Control: best threshold = 0.15, F2 (val) = 0.7899
XGBoost Weighted: best threshold = 0.93, F2 (val) = 0.8214
LightGBM Control: best threshold = 0.08, F2 (val) = 0.7545


In [18]:
from sklearn.metrics import f1_score, matthews_corrcoef

models_test = {
    'XGBoost Control': (xgb_control, thresholds['XGBoost Control']),
    'XGBoost Weighted': (xgb_weighted, thresholds['XGBoost Weighted']),
    'LightGBM Control': (lgbm_control, thresholds['LightGBM Control']),
}

results = []
for name, (model, t) in models_test.items():
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= t).astype(int)

    pr_auc = average_precision_score(y_test, probs)
    f1 = f1_score(y_test, preds, zero_division=0)
    f2 = fbeta_score(y_test, preds, beta=2, zero_division=0)
    mcc = matthews_corrcoef(y_test, preds)

    results.append((name, t, pr_auc, f1, f2, mcc))
    print(f'{name} (threshold={t:.2f}): PR-AUC={pr_auc:.4f}, F1={f1:.4f}, F2={f2:.4f}, MCC={mcc:.4f}')

XGBoost Control (threshold=0.15): PR-AUC=0.7467, F1=0.6630, F2=0.7437, MCC=0.6726
XGBoost Weighted (threshold=0.93): PR-AUC=0.7954, F1=0.6668, F2=0.7578, MCC=0.6791
LightGBM Control (threshold=0.08): PR-AUC=0.5743, F1=0.5734, F2=0.6885, MCC=0.5951
